In [ ]:
import requests
import psycopg2
from datetime import datetime

cities = {
    "New York": (40.7128, -74.0060),
    "Los Angeles": (34.0522, -118.2437),
    "Chicago": (41.8781, -87.6298),
    "Houston": (29.7604, -95.3698),
    "Miami": (25.7617, -80.1918),
    "London": (51.5074, -0.1278),
    "Paris": (48.8566, 2.3522),
    "Berlin": (52.5200, 13.4050),
    "Tokyo": (35.6895, 139.6917),
    "Osaka": (34.6937, 135.5023),
    "Beijing": (39.9042, 116.4074),
    "Shanghai": (31.2304, 121.4737),
    "Mumbai": (19.0760, 72.8777),
    "Bengaluru": (12.9716, 77.5946),
    "Delhi": (28.7041, 77.1025),
    "Sydney": (-33.8688, 151.2093),
    "Melbourne": (-37.8136, 144.9631),
    "Toronto": (43.6532, -79.3832),
    "Vancouver": (49.2827, -123.1207),
    "Moscow": (55.7558, 37.6173),
    "Dubai": (25.2048, 55.2708),
    "Singapore": (1.3521, 103.8198),
    "Johannesburg": (-26.2041, 28.0473),
    "Rio de Janeiro": (-22.9068, -43.1729),
}

DB_NAME = "weather"
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "localhost"
DB_PORT = "5432"

conn = psycopg2.connect(
    dbname=DB_NAME,
    user=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT
)
cursor = conn.cursor()
for city, (lat, lon) in cities.items():
    url = f"https://api.open-meteo.com/v1/forecast?latitude={lat}&longitude={lon}&hourly=temperature_2m,relativehumidity_2m"
    res = requests.get(url).json()

    timestamps = res['hourly']['time']
    temps = res['hourly']['temperature_2m']
    hums = res['hourly'].get('relativehumidity_2m') or [None]*len(timestamps)

    for t, temp, hum in zip(timestamps, temps, hums):
        cursor.execute(
            """
            INSERT INTO weather_raw (city, latitude, longitude, timestamp, temperature, humidity)
            VALUES (%s, %s, %s, %s, %s, %s)
            ON CONFLICT DO NOTHING
            """
            (city, lat, lon, datetime.fromisoformat(t), temp, hum)
        )

conn.commit()
cursor.close()
conn.close()

print(" Weather data inserted into weather_raw table for all cities")


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col , avg ,  min , max , to_date

In [ ]:
spark = SparkSession.builder.appName("weather_etl").config("spark.jars.packages", "org.postgresql:postgresql:42.2.18").getOrCreate()

In [ ]:
spark

In [ ]:
jdbc_url = "jdbc:postgresql://localhost:5432/weather"
conn_info = {
    "user" : "postgres",
    "password" : "postgres" , 
    "driver" : "org.postgresql.Driver"
}

In [ ]:
df = spark.read.jdbc(url=jdbc_url , table="weather_raw" , properties=conn_info)

In [ ]:
df.show()

In [ ]:

df_daily = (
    df.withColumn("date", to_date(col("timestamp")))
          .groupBy("city", "date")
          .agg(
              avg("temperature").alias("avg_temperature"),
              min("temperature").alias("min_temperature"),
              max("temperature").alias("max_temperature"),
              avg("humidity").alias("avg_humidity")
          )
)

In [ ]:
df_daily.show()

In [ ]:
from pyspark.sql.window import Window
from pyspark.sql.functions import rank
window = Window.partitionBy("date").orderBy(col("avg_temperature").desc())
df_daily = df_daily.withColumn("city_rank_by_temp", rank().over(window))

In [ ]:
df_daily.write.jdbc(url = jdbc_url , table = "weather_clean" , mode = "overwrite", properties= conn_info)
print("ETL process completed: weather_clean table updated.")